# S08 · Prefill/decode disaggregation: is the KV transfer worth it?

Part of **[Inference Engineering](https://github.com/supriyo100/kv-cache-attention-variants)**, an original notebook (not from the upstream course).
Derivation: path step 12. Sources: DistServe (arXiv:2401.09670); Mooncake (arXiv:2407.00079, github.com/kvcache-ai/Mooncake).

Every number below is produced by code in this notebook or in `inference_lab` (tested in
`tests/`). Labels: *measured* = ran here (CPU), *theoretical* = formula with datasheet inputs.

In [ ]:
# Setup: works locally (uv sync) and on Colab (installs the small inference_lab package only)
try:
    import inference_lab  # noqa: F401
except ImportError:  # Colab / fresh environment
    %pip install -q --no-deps "git+https://github.com/supriyo100/kv-cache-attention-variants"
import torch
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (7.5, 3.6), "axes.spines.top": False, "axes.spines.right": False})
C = {"compute": "#0E7C72", "memory": "#B5790A", "waste": "#B42B3E", "shared": "#5B47C9", "ink": "#1C232B"}

## Objective
Quantify the KV transfer that disaggregation adds, relative to the prefill it follows.

## Theory and equations
$t_{\text{xfer}} = T_p m_{\text{tok}}/\beta_{\text{link}}$; ratio $t_{\text{xfer}}/t_{\text{prefill}} \propto m_{\text{tok}}/P$ (prompt length cancels when prefill is compute-bound).

## Diagram
```
[prefill pool] --KV (T_p x m_tok bytes) over RDMA--> [decode pool] --> tokens
   sets TTFT                                           sets ITL
```

## Implementation

In [ ]:
from inference_lab import disagg as D, kv_math as K
models = [K.LLAMA3_8B, K.LLAMA3_70B, K.DEEPSEEK_V3]

## Experiment (*theoretical*)

In [ ]:
for m in models:
    print(m.name, f"prefill 8K on H100 roofline: {1e3*D.prefill_time(m, K.H100_SXM, 8192):.0f} ms")
    for link, bw in D.LINKS_GBS.items():
        print(f"   {link:22s} transfer {1e3*D.transfer_time(m, 8192, bw):7.1f} ms   ratio {D.transfer_ratio(m, K.H100_SXM, 8192, bw):.3f}")

## Results: ratio vs prompt length

In [ ]:
Ts = [256, 1024, 4096, 16384, 65536]
for m, col in zip(models, [C["waste"], C["memory"], C["compute"]]):
    plt.plot(Ts, [D.transfer_ratio(m, K.H100_SXM, T, 50.0) for T in Ts], "o-", color=col, label=m.name)
plt.xscale("log", base=2); plt.yscale("log"); plt.axhline(0.1, color=C["ink"], ls="--", lw=1)
plt.xlabel("prompt tokens"); plt.ylabel("transfer / prefill (IB NDR 400G)"); plt.legend(); plt.show()

## Interpretation
Big models with compact caches (GQA-8 70B, MLA) make transfer a few percent of prefill on RDMA.
Small models on slow links do not. Short prompts are memory-bound in prefill, which raises the
ratio again.

## Limitations
Roofline prefill (an optimistic lower bound on prefill time makes ratios pessimistic); no
layer-wise overlap, which real systems use to hide transfer.

## Conclusion
Disaggregate large models on fast fabrics; keep small models co-located with chunked prefill.